In [1]:
# FULL SETUP — clone, fix every known dependency/code issue, download checkpoints

!git clone https://github.com/levihsu/OOTDiffusion.git /content/OOTDiffusion
%cd /content/OOTDiffusion

!sed -E -i 's/==[0-9][^ ]*//' requirements.txt
!pip install -q -r requirements.txt
!pip install -q basicsr

# Patch 1: basicsr's broken torchvision import
!sed -i 's/from torchvision.transforms.functional_tensor import rgb_to_grayscale/from torchvision.transforms.functional import rgb_to_grayscale/' \
/usr/local/lib/python3.12/dist-packages/basicsr/data/degradations.py

# Patch 2: CaptionProjection renamed in newer diffusers
!grep -rl "from diffusers.models.embeddings import CaptionProjection" ./ootd | xargs -r sed -i \
's/from diffusers.models.embeddings import CaptionProjection/from diffusers.models.embeddings import PixArtAlphaTextProjection as CaptionProjection, PatchEmbed/'

# Patch 3: DualTransformer2DModel moved in newer diffusers
!grep -rl "from diffusers.models.dual_transformer_2d import DualTransformer2DModel" ./ootd | xargs -r sed -i \
's/from diffusers.models.dual_transformer_2d import DualTransformer2DModel/from diffusers.models.transformers.dual_transformer_2d import DualTransformer2DModel/'

# Patch 4: PositionNet removed entirely from newer diffusers (dead GLIGEN code OOTDiffusion never actually uses — safe to drop import)
!grep -rl "^    PositionNet,$" ./ootd | xargs -r sed -i '/^    PositionNet,$/d'

# Patch 5: remove variant="fp16" (checkpoints aren't in that format) + clean orphaned commas
!grep -rl 'variant="fp16"' ./ootd | xargs -r sed -i 's/, variant="fp16"//g; s/variant="fp16", //g'
!sed -i '/^[[:space:]]*,[[:space:]]*$/d' ./ootd/inference_ootd_hd.py ./ootd/inference_ootd_dc.py ./ootd/inference_ootd.py

# Patch 6: remove use_safetensors=True (some checkpoints are .bin, not .safetensors)
!grep -rl "use_safetensors=True" ./ootd | xargs -r sed -i '/use_safetensors=True/d'

# Patch 7: force single GPU (device 0) — free Colab only gives us one
!sed -i \
-e 's/OpenPose(1)/OpenPose(0)/' \
-e 's/Parsing(1)/Parsing(0)/' \
-e 's/OOTDiffusionDC(1)/OOTDiffusionDC(0)/' \
run/gradio_ootd.py

# Patch 8: enable public sharing
!sed -i 's/block.launch(server_name="0.0.0.0", server_port=7865)/block.launch(server_name="0.0.0.0", server_port=7865, share=True)/' \
run/gradio_ootd.py

!find ./ootd -name "__pycache__" -exec rm -rf {} +

# Download checkpoints — straight into checkpoints/, no wasteful temp-copy step
from huggingface_hub import snapshot_download
snapshot_download(repo_id="levihsu/OOTDiffusion", local_dir="checkpoints_raw")
!mv checkpoints_raw/checkpoints/* checkpoints/
!rm -rf checkpoints_raw

# CLIP model — restricted to only the format actually used, saves ~4GB
snapshot_download(
    repo_id="openai/clip-vit-large-patch14",
    local_dir="checkpoints/clip-vit-large-patch14",
    allow_patterns=["*.safetensors", "*.json", "*.txt"],
)

print("SETUP COMPLETE")
!ls checkpoints


Cloning into '/content/OOTDiffusion'...
remote: Enumerating objects: 800, done.
remote: Counting objects: 100% (78/78), done.
remote: Compressing objects: 100% (50/50), done.
remote: Total 800 (delta 46), reused 28 (delta 28), pack-reused 722 (from 1)
Receiving objects: 100% (800/800), 27.30 MiB | 21.54 MiB/s, done.
Resolving deltas: 100% (180/180), done.
/content/OOTDiffusion
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 83.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.5/172.5 kB 7.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 344.7/344.7 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 108.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.2/256.2 kB 18.1 MB/s eta 0:00:00


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 28 files:   0%|          | 0/28 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

SETUP COMPLETE
clip-vit-large-patch14	humanparsing  ootd  openpose  README.txt


In [2]:
import re

files = [
    "/content/OOTDiffusion/ootd/inference_ootd_hd.py",
    "/content/OOTDiffusion/ootd/inference_ootd_dc.py",
    "/content/OOTDiffusion/ootd/inference_ootd.py",
]

for f in files:
    with open(f, "r") as fh:
        content = fh.read()
    # Remove variant="fp16" or variant='fp16' in any position: leading comma, trailing comma, or alone on its own line
    new_content = re.sub(r',?\s*variant=["\']fp16["\']\s*,?', "", content)
    if new_content != content:
        with open(f, "w") as fh:
            fh.write(new_content)
        print(f"Patched: {f}")
    else:
        print(f"No match found in: {f}")

# Confirm nothing's left
!grep -rn 'variant="fp16"\|variant='"'"'fp16'"'"'' /content/OOTDiffusion/ootd

Patched: /content/OOTDiffusion/ootd/inference_ootd_hd.py
Patched: /content/OOTDiffusion/ootd/inference_ootd_dc.py
Patched: /content/OOTDiffusion/ootd/inference_ootd.py


In [3]:
!sed -n '45,65p' /content/OOTDiffusion/ootd/inference_ootd_hd.py

            subfolder="unet_garm",
            torch_dtype=torch.float16,
        )
        unet_vton = UNetVton2DConditionModel.from_pretrained(
            UNET_PATH,
            subfolder="unet_vton",
            torch_dtype=torch.float16,
        )

        self.pipe = OotdPipeline.from_pretrained(
            MODEL_PATH,
            unet_garm=unet_garm,
            unet_vton=unet_vton,
            vae=vae,
            torch_dtype=torch.float16
            safety_checker=None,
            requires_safety_checker=False,
        ).to(self.gpu_id)

        self.pipe.scheduler = UniPCMultistepScheduler.from_config(self.pipe.scheduler.config)
        


In [4]:
import re

files = [
    "/content/OOTDiffusion/ootd/inference_ootd_hd.py",
    "/content/OOTDiffusion/ootd/inference_ootd_dc.py",
    "/content/OOTDiffusion/ootd/inference_ootd.py",
]

for f in files:
    with open(f, "r") as fh:
        content = fh.read()
    # Insert the missing comma between torch_dtype=torch.float16 and the next argument
    new_content = re.sub(
        r'(torch_dtype=torch\.float16)(\s*\n\s*)(safety_checker=None)',
        r'\1,\2\3',
        content
    )
    if new_content != content:
        with open(f, "w") as fh:
            fh.write(new_content)
        print(f"Fixed: {f}")
    else:
        print(f"No match (may already be correct): {f}")

Fixed: /content/OOTDiffusion/ootd/inference_ootd_hd.py
Fixed: /content/OOTDiffusion/ootd/inference_ootd_dc.py
Fixed: /content/OOTDiffusion/ootd/inference_ootd.py


In [5]:
!sed -n '55,62p' /content/OOTDiffusion/ootd/inference_ootd_hd.py

            MODEL_PATH,
            unet_garm=unet_garm,
            unet_vton=unet_vton,
            vae=vae,
            torch_dtype=torch.float16,
            safety_checker=None,
            requires_safety_checker=False,
        ).to(self.gpu_id)


In [6]:
!grep -n "block.launch" /content/OOTDiffusion/run/gradio_ootd.py

260:block.launch(server_name='0.0.0.0', server_port=7865)


In [7]:
import re

f = "/content/OOTDiffusion/run/gradio_ootd.py"
with open(f, "r") as fh:
    content = fh.read()

old = "block.launch(server_name='0.0.0.0', server_port=7865)"
new = "block.launch(server_name='0.0.0.0', server_port=7865, share=True)"

if old in content:
    content = content.replace(old, new)
    with open(f, "w") as fh:
        fh.write(content)
    print("Patched successfully")
else:
    print("Exact line not found — check formatting")

!grep -n "block.launch" /content/OOTDiffusion/run/gradio_ootd.py

Patched successfully
260:block.launch(server_name='0.0.0.0', server_port=7865, share=True)


In [ ]:
%cd /content/OOTDiffusion/run
!python gradio_ootd.py

/content/OOTDiffusion/run
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
An error occurred while trying to fetch ../checkpoints/ootd: Error no file named diffusion_pytorch_model.safetensors found in directory ../checkpoints/ootd.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
Loading pipeline components...:   0% 0/7 [00:00<?, ?it/s]
Loading weights:   0% 0/196 [00:00<?, ?it/s]
Loading weights:   1% 2/196 [00:01<01:40,  1.93it/s]
Loading weights:  45% 88/196 [00:01<00:01, 104.50it/s]
Loading weights:  68% 133/196 [00:01<00:00, 124.67it/s]
Loading weights:  85% 166/196 [00:01<00:00, 131.26it/s]
Loading weights: 100% 196/196 [00:01<00:00, 105.04it/s]
Loading pipeline components...: 100% 7/7 [00:02<0